In [0]:
import numpy as np
from sklearn.manifold import MDS, LocallyLinearEmbedding, TSNE
from src.visualization import plot_dim_reduction

In [0]:
# Load clustered results from gold layer
df_label = spark.table("scientific_trend_analysis.rep.arxiv_clustered").toPandas()
print(f"Loaded {len(df_label)} records with {len(df_label['Cluster_ID_km'].unique())} clusters")
print(f"Loaded {len(df_label)} records with {len(df_label['Cluster_ID_gmm'].unique())} clusters")
display(df_label.head())

#### K-Means Visuals

In [0]:
# Load LSA features from volume
import pickle
with open('/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_features_nor.pkl', 'rb') as f:
    lsa_result_nor = pickle.load(f)

# Extract cluster labels from the loaded dataframe
cluster_labels_km = df_label['Cluster_ID_km'].values

print(f"Loaded LSA features: {lsa_result_nor.shape}")

In [0]:
sample_fraction = 0.05
stratified_sample_km = df_label.groupby('Cluster_ID_km') \
                            .apply(lambda x: x.sample(frac=sample_fraction, random_state=2), include_groups=False)
sample_size = len(stratified_sample_km)

print(f'Sampled {sample_fraction*100}% of each cluster, total rows: {sample_size}\n')
print(df_label['Cluster_ID_km'].value_counts(normalize=True))

stratified_indices_km = stratified_sample_km.index.get_level_values(1).values
lsa_stratified_km = lsa_result_nor[stratified_indices_km]

# Stratified samples
cluster_labels_numpy_km = np.array(cluster_labels_km)
km_sample_labels = cluster_labels_numpy_km[stratified_indices_km]
print(f"\nStratified LSA features extracted for visualization: {lsa_stratified_km.shape}")

In [0]:
# Run MDS on stratified LSA sample
mds = MDS(
    n_components=2, 
    random_state=2, 
    n_init=1, 
    max_iter=300
)
mds_2d_km = mds.fit_transform(lsa_stratified_km)
df_plot_mds_km = plot_dim_reduction(mds_2d_km, km_sample_labels, 'MDS', 'Cluster_ID_km', sample_size, 'KMeans')

print(f"Stress: {mds.stress_:.2f}")
print(f"Converged in {mds.n_iter_} iterations")

In [0]:
# Try multiple n_neighbors values for LLE and select the best based on reconstruction error
n_neighbors_list = [5, 10, 15, 20, 30, 50, 75, 100]
lle_errors = {}
lle_results = {}

for n in n_neighbors_list:
    lle = LocallyLinearEmbedding(
        n_components=2, 
        random_state=2, 
        n_neighbors=n
    )
    lle_2d = lle.fit_transform(lsa_stratified_km)
    lle_errors[n] = lle.reconstruction_error_
    lle_results[n] = lle_2d
    print(f"n_neighbors={n}, Reconstruction Error: {lle.reconstruction_error_:.8f}")

best_n = min(lle_errors, key=lle_errors.get)
lle_2d_km = lle_results[best_n]
df_plot_lle_km = plot_dim_reduction(lle_2d_km, km_sample_labels, 'LLE', 'Cluster_ID_km', sample_size, 'KMeans')

print(f"Best n_neighbors: {best_n} with Reconstruction Error: {lle_errors[best_n]:.8f}")

In [0]:
# Use stratified sample for t-SNE visualization
tsne = TSNE(
    n_components=2, 
    random_state=2,
    perplexity=50,
    init='pca'
)
tsne_2d_km = tsne.fit_transform(lsa_stratified_km)
df_plot_tsne_km = plot_dim_reduction(tsne_2d_km, km_sample_labels, 't-SNE', 'Cluster_ID_km', sample_size, 'KMeans')

print(f"KL Divergence: {tsne.kl_divergence_:.4f}")

#### GMM Visuals

In [0]:
# Load LSA features from volume
import pickle
with open('/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_features.pkl', 'rb') as f:
    lsa_result = pickle.load(f)

# Extract cluster labels from the loaded dataframe
cluster_labels_gmm = df_label['Cluster_ID_gmm'].values

print(f"Loaded LSA features: {lsa_result.shape}")

In [0]:
sample_fraction = 0.05
stratified_sample_gmm = df_label.groupby('Cluster_ID_gmm') \
                            .apply(lambda x: x.sample(frac=sample_fraction, random_state=2), include_groups=False)
sample_size = len(stratified_sample_gmm)

print(f'Sampled {sample_fraction*100}% of each cluster, total rows: {sample_size}\n')
print(df_label['Cluster_ID_gmm'].value_counts(normalize=True))

stratified_indices_gmm = stratified_sample_gmm.index.get_level_values(1).values
lsa_stratified_gmm = lsa_result[stratified_indices_gmm]

# Stratified samples
cluster_labels_numpy_gmm = np.array(cluster_labels_gmm)
gmm_sample_labels = cluster_labels_numpy_gmm[stratified_indices_gmm]
print(f"\nStratified LSA features extracted for visualization: {lsa_stratified_gmm.shape}")

In [0]:
mds = MDS(
    n_components=2, 
    random_state=2, 
    n_init=1,
    max_iter=300
)
mds_2d_gmm = mds.fit_transform(lsa_stratified_gmm)
df_plot_mds_gmm = plot_dim_reduction(mds_2d_gmm, gmm_sample_labels, 'MDS', 'Cluster_ID_gmm', sample_size, 'GaussianMixture')

print(f"Stress: {mds.stress_:.2f}")
print(f"Converged in {mds.n_iter_} iterations")

In [0]:
# Try multiple n_neighbors values for LLE and select the best based on reconstruction error
n_neighbors_list = [5, 10, 15, 20, 30, 50, 75, 100]
lle_errors = {}
lle_results = {}

for n in n_neighbors_list:
    lle = LocallyLinearEmbedding(
        n_components=2, 
        random_state=2, 
        n_neighbors=n
    )
    lle_2d = lle.fit_transform(lsa_stratified_gmm)
    lle_errors[n] = lle.reconstruction_error_
    lle_results[n] = lle_2d
    print(f"n_neighbors={n}, Reconstruction Error: {lle.reconstruction_error_:.8f}")

best_n = min(lle_errors, key=lle_errors.get)
lle_2d_km = lle_results[best_n]
df_plot_lle_gmm = plot_dim_reduction(lle_2d_gmm, gmm_sample_labels, 'LLE', 'Cluster_ID_gmm', sample_size, 'GaussianMixture')

print(f"Best n_neighbors: {best_n} with Reconstruction Error: {lle_errors[best_n]:.8f}")

In [0]:
tsne = TSNE(
    n_components=2, 
    random_state=2,
    perplexity=50,
    learning_rate=200,
    early_exaggeration=12,
    init='pca'
)
tsne_2d_gmm = tsne.fit_transform(lsa_stratified_gmm)
df_plot_tsne_gmm = plot_dim_reduction(tsne_2d_gmm, gmm_sample_labels, 't-SNE', 'Cluster_ID_gmm', sample_size, 'GaussianMixture')

print(f"KL Divergence: {tsne.kl_divergence_:.4f}")